# Приоритизация обращений

Ранжирование назначенных обращений по вероятности успешного целевого действия в течение 5 дней.
Метрика — **Daily Average Precision**: AP считается внутри каждой даты назначения и усредняется по дням.

Логика вынесена в пакет `src/`, ноутбук выполняет роль сценария запуска:

| Модуль | Назначение |
|---|---|
| `src/config.py` | константы схемы данных, сиды, параметры валидации |
| `src/metrics.py` | Average Precision и Daily Average Precision |
| `src/features.py` | признаки из событий, производные отношения, скользящие перцентили, согласование категорий |
| `src/validation.py` | разбиение по дням с расширяющимся окном |
| `src/models.py` | обёртки LightGBM / CatBoost / XGBoost |
| `src/blending.py` | ранговое смешивание, жадный отбор, стекинг |

Запуск: `pip install -r requirements.txt`, затем выполнить ноутбук сверху вниз. На выходе — `submission.csv`.

## 1. Окружение

In [8]:
import sys
from pathlib import Path

# Пакет src лежит рядом с ноутбуком; на Kaggle каталог решения подключается как dataset.
for candidate in [Path.cwd(), Path.cwd().parent, Path('/kaggle/input/datasets/valerakarmanov/avito-solution/AVITO_~1')]:
    if (candidate / 'src' / 'config.py').exists():
        sys.path.insert(0, str(candidate))
        break

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from src import config, features, models, blending
from src.metrics import daily_average_precision
from src.validation import TimeSeriesDaySplit

optuna.logging.set_verbosity(optuna.logging.WARNING)
np.random.seed(config.SEED)

## 2. Загрузка данных

In [9]:
def resolve(candidates, filename=None):
    """Первый существующий путь из списка кандидатов."""
    for path in candidates:
        target = path / filename if filename else path
        if target.exists():
            return path if filename else target
    raise FileNotFoundError(f'не найдено: {filename or candidates}')

DATA_DIR = resolve(config.DATA_DIR_CANDIDATES, 'train.csv')
SAMPLE_PATH = resolve(config.SAMPLE_SUBMISSION_CANDIDATES)

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
events = pd.read_csv(DATA_DIR / 'events.csv')
sample_submission = pd.read_csv(SAMPLE_PATH)

for frame in (train, test):
    frame['assignment_ts'] = pd.to_datetime(frame['assignment_ts'])
events['event_ts'] = pd.to_datetime(events['event_ts'])

print(f'train {train.shape} | test {test.shape} | events {events.shape}')
print(f"доля положительных: {train[config.TARGET].mean():.4f}")
print(f"train: {train['assignment_date'].min()} .. {train['assignment_date'].max()}")
print(f"test:  {test['assignment_date'].min()} .. {test['assignment_date'].max()}")

train (13694, 119) | test (4306, 118) | events (254705, 7)
доля положительных: 0.2075
train: 2026-04-07 .. 2026-04-22
test:  2026-04-23 .. 2026-04-27


## 3. Признаки

Три группы:

1. **Из `events.csv`** — агрегаты поведения пользователя: объём и давность активности, счётчики и
   давность по типам событий, окна 1/2/3/7/14/30 дней, доля high-intent действий, интервалы между
   событиями, время суток, траектория цены, гистограмма контекстов, тип последнего события.
   Учитываются только события с `event_ts < assignment_ts`.
2. **Производные отношения** поверх готовых агрегатов: ускорение активности и конверсия прошлых обращений.
3. **Скользящие перцентили** — позиция обращения относительно назначенных в предыдущие 24 часа.
   Метрика сравнивает обращения внутри дня, поэтому относительная позиция несёт информацию,
   которой нет в абсолютных значениях. Используются только более ранние назначения.

Перцентили вынесены в отдельный набор `FEATURES_WITH_TRAILING`: одна из моделей пула обучается
на нём, остальные — на базовом наборе, что добавляет ансамблю разнообразия.

In [10]:
meta = pd.concat([train[['lead_id', 'assignment_ts']], test[['lead_id', 'assignment_ts']]])
event_features = features.build_event_features(events, meta)
train = train.merge(event_features, on='lead_id', how='left')
test = test.merge(event_features, on='lead_id', how='left')

train = features.add_ratio_features(train)
test = features.add_ratio_features(test)

# Перцентили считаются по объединённой выборке: окно опирается на время назначения,
# а не на принадлежность к train или test, и смотрит только в прошлое.
combined = pd.concat([train.assign(_part='train'), test.assign(_part='test')], ignore_index=True)
combined, TRAILING_COLUMNS = features.add_trailing_percentiles(combined)
train = combined[combined['_part'] == 'train'].drop(columns='_part').reset_index(drop=True)
test = combined[combined['_part'] == 'test'].drop(columns=['_part', config.TARGET]).reset_index(drop=True)

print(f'признаков из событий: {event_features.shape[1] - 1}')

признаков из событий: 65


## 4. Категориальные признаки и пропуски

* **Категории.** Хранятся как `category`: LightGBM и XGBoost работают с ними напрямую,
  CatBoost получает индексы колонок и применяет упорядоченное кодирование.
  One-hot не используется — он теряет информацию о частоте и раздувает пространство.
* **Редкие значения.** Категории с частотой в train ниже `MIN_CATEGORY_COUNT` и значения,
  не встречавшиеся в train, схлопываются в общий токен. На текущих данных ни одна категория
  под порог не попадает, но правило защищает от незнакомых значений в скрытом тесте.
* **Пропуски.** Не заполняются. Бустинги выбирают для NaN направление ветвления,
  что информативнее подстановки медианы: пропуск здесь означает отсутствие истории
  (например, обращение без предшествующих событий), и это само по себе сигнал.
  Исключение — категориальные колонки для CatBoost, который требует явного значения.

In [11]:
CATEGORICAL = config.BASE_CATEGORICAL + ['ev_last_type', 'ev_last_ctx']
train, test = features.harmonize_categories(train, test, CATEGORICAL)

EXCLUDED = set(config.ID_COLUMNS + config.TIME_COLUMNS + [config.TARGET]) | set(TRAILING_COLUMNS)
FEATURES = [c for c in train.columns if c not in EXCLUDED]
FEATURES_WITH_TRAILING = FEATURES + TRAILING_COLUMNS

missing_share = train[FEATURES].isna().mean()
print(f'признаков: {len(FEATURES)} (с перцентилями {len(FEATURES_WITH_TRAILING)})')
print(f'колонок с пропусками: {(missing_share > 0).sum()}, максимальная доля {missing_share.max():.3f}')

признаков: 189 (с перцентилями 201)
колонок с пропусками: 168, максимальная доля 0.789


## 5. Валидация

Тест лежит строго позже train по времени, поэтому случайное разбиение завышало бы качество:
модель видела бы «будущие» дни. Используется time-series CV с расширяющимся окном — обучение
всегда на днях, предшествующих валидационному блоку.

Шесть блоков по два дня покрывают последние 12 дней train. Блоки по два дня, а не по одному,
потому что Daily AP на одном дне слишком шумна; 12 дней покрытия дают достаточно данных
для устойчивого подбора весов ансамбля.

In [12]:
day = pd.to_datetime(train['assignment_date']).dt.date
split = TimeSeriesDaySplit(day=day, n_validation_days=config.N_VALIDATION_DAYS,
                           block_size=config.VALIDATION_BLOCK)

y = train[config.TARGET].values.astype(float)
assignment_date = train['assignment_date'].values
oof_mask = split.oof_mask
oof_y, oof_dates = y[oof_mask], assignment_date[oof_mask]

model_data = models.ModelData(train=train, test=test, y=y, features=FEATURES,
                              categorical=CATEGORICAL, split=split,
                              assignment_date=assignment_date)

print(f'фолдов: {len(split.folds)}, строк в OOF: {oof_mask.sum()}')
for block in split.folds:
    print(f'  валидация {block[0]} .. {block[-1]}')

фолдов: 6, строк в OOF: 10300
  валидация 2026-04-11 .. 2026-04-12
  валидация 2026-04-13 .. 2026-04-14
  валидация 2026-04-15 .. 2026-04-16
  валидация 2026-04-17 .. 2026-04-18
  валидация 2026-04-19 .. 2026-04-20
  валидация 2026-04-21 .. 2026-04-22


## 6. Подбор гиперпараметров LightGBM

Optuna максимизирует Daily AP на отложенных последних трёх днях train. Проход по всему CV
внутри каждой попытки был бы в шесть раз дороже и не позволил бы перебрать достаточно
конфигураций, а найденные параметры затем всё равно честно оцениваются на полном CV.

В пул берутся две конфигурации: лучшая и лучшая из существенно отличающихся по глубине или
числу листьев. Вторая нужна не ради качества сама по себе, а ради декорреляции ошибок в ансамбле.

In [13]:
holdout_days = split.unique_days[-3:]
holdout_mask = day.isin(holdout_days).values
fit_mask = ~holdout_mask

# feature_pre_filter отключён: Optuna меняет min_child_samples, а предварительная
# фильтрация признаков жёстко привязывает Dataset к первому значению этого параметра.
dataset_params = {'feature_pre_filter': False}
dtrain_ho = lgb.Dataset(train.loc[fit_mask, FEATURES], y[fit_mask],
                        categorical_feature=CATEGORICAL, params=dataset_params)
dvalid_ho = lgb.Dataset(train.loc[holdout_mask, FEATURES], y[holdout_mask],
                        categorical_feature=CATEGORICAL, reference=dtrain_ho, params=dataset_params)

def daily_ap_eval(preds, dataset):
    return 'daily_ap', daily_average_precision(dataset.get_label(), preds, assignment_date[holdout_mask]), True

def objective(trial):
    params = dict(
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 80),
        min_child_samples=trial.suggest_int('min_child_samples', 15, 150),
        feature_fraction=trial.suggest_float('feature_fraction', 0.4, 0.95),
        bagging_fraction=trial.suggest_float('bagging_fraction', 0.5, 0.95),
        bagging_freq=1,
        lambda_l1=trial.suggest_float('lambda_l1', 1e-3, 20, log=True),
        lambda_l2=trial.suggest_float('lambda_l2', 1e-3, 20, log=True),
        max_depth=trial.suggest_int('max_depth', 3, 10),
    )
    booster = lgb.train(
        {**params, 'objective': 'binary', 'metric': 'None', 'verbose': -1,
         'seed': config.SEED, 'feature_pre_filter': False},
        dtrain_ho, 1200, valid_sets=[dvalid_ho], feval=daily_ap_eval,
        callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False)],
    )
    return booster.best_score['valid_0']['daily_ap']

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=config.SEED))
study.optimize(objective, n_trials=config.N_OPTUNA_TRIALS, show_progress_bar=False)

ranked = sorted([t for t in study.trials if t.value is not None], key=lambda t: -t.value)
primary_params = dict(ranked[0].params, bagging_freq=1)
alternative = next(
    (t for t in ranked[1:]
     if abs(t.params['num_leaves'] - primary_params['num_leaves']) > 15
     or abs(t.params['max_depth'] - primary_params['max_depth']) > 2),
    ranked[min(1, len(ranked) - 1)],
)
alternative_params = dict(alternative.params, bagging_freq=1)

print(f'holdout Daily AP: основная {ranked[0].value:.5f}, альтернативная {alternative.value:.5f}')

holdout Daily AP: основная 0.71831, альтернативная 0.71741


## 7. Пул моделей

Девять моделей трёх семейств. Разнообразие важнее индивидуального качества: жадный отбор
на следующем шаге сам решит, кого включить, а бесполезные модели получат вес 0.

Ранжирующие постановки (`rank:map`, `lambdarank`) сгруппированы по дню назначения — это
прямое соответствие метрике. Их результат приводится ниже как есть: на данном объёме дней
они уступают классификаторам, и отбор это учитывает.

In [14]:
pool = {}

def add_model(name, fn):
    """Обучает модель и печатает её OOF Daily AP.

    Ошибка отдельной модели (например, несовместимость версии библиотеки) не должна
    рушить весь прогон: такая модель исключается из пула, состав пула печатается.
    """
    try:
        oof, test_pred = fn()
    except Exception as error:
        print(f'{name:12s} исключена: {type(error).__name__}: {error}')
        return
    pool[name] = (oof, test_pred)
    print(f'{name:12s} OOF Daily AP: {daily_average_precision(oof_y, oof[oof_mask], oof_dates):.5f}')

add_model('lgb_primary', lambda: models.run_lightgbm(model_data, primary_params))
add_model('lgb_alt', lambda: models.run_lightgbm(model_data, alternative_params))
add_model('lgb_trailing', lambda: models.run_lightgbm(model_data, primary_params,
                                                      features=FEATURES_WITH_TRAILING))
add_model('lgb_dart', lambda: models.run_lightgbm(
    model_data, dict(primary_params, learning_rate=max(primary_params['learning_rate'], 0.05)),
    boosting='dart'))
add_model('catboost_d6', lambda: models.run_catboost(model_data, depth=6))
add_model('catboost_d4', lambda: models.run_catboost(model_data, depth=4))
add_model('xgb_clf', lambda: models.run_xgboost(model_data, 'binary:logistic'))
add_model('xgb_rank', lambda: models.run_xgboost(model_data, 'rank:map'))
add_model('lgb_rank', lambda: models.run_lightgbm_ranker(model_data))

print(f'\nмоделей в пуле: {len(pool)}')

lgb_primary  OOF Daily AP: 0.68136
lgb_alt      OOF Daily AP: 0.67821
lgb_trailing OOF Daily AP: 0.68200
lgb_dart     OOF Daily AP: 0.67039
catboost_d6  OOF Daily AP: 0.67739
catboost_d4  OOF Daily AP: 0.67836
xgb_clf      исключена: TypeError: Invalid type for the `evals`.
xgb_rank     исключена: TypeError: Invalid type for the `evals`.
lgb_rank     OOF Daily AP: 0.56099

моделей в пуле: 7


## 8. Смешивание

Сравниваются два способа: жадный отбор с возвратом (метод Каруаны) и стекинг логистической
регрессией. Стекинг оценивается по схеме leave-one-fold-out — обучать мета-модель на тех же
OOF, на которых её оценивают, значит получить смещённую оценку.

Побеждает вариант с более высоким OOF Daily AP.

In [15]:
names = list(pool)
oof_ranks = np.vstack([blending.to_rank(pool[n][0][oof_mask]) for n in names])
test_ranks = np.vstack([blending.to_rank(pool[n][1]) for n in names])
fold_ids = split.fold_ids[oof_mask]

single_scores = [daily_average_precision(oof_y, oof_ranks[i], oof_dates) for i in range(len(names))]
greedy_weights, greedy_score = blending.greedy_ensemble(oof_ranks, oof_y, oof_dates)
stack_score = blending.stacking_score(oof_ranks, oof_y, oof_dates, fold_ids)

best_single = int(np.argmax(single_scores))
print(f'лучшая одиночная: {single_scores[best_single]:.5f} ({names[best_single]})')
print(f'жадный отбор:     {greedy_score:.5f}')
print(f'стекинг (LOFO):   {stack_score:.5f}')
print('веса:', {names[i]: round(float(greedy_weights[i]), 3)
                for i in range(len(names)) if greedy_weights[i] > 0})

лучшая одиночная: 0.68200 (lgb_trailing)
жадный отбор:     0.68777
стекинг (LOFO):   0.68444
веса: {'lgb_primary': 0.167, 'lgb_alt': 0.167, 'lgb_trailing': 0.333, 'catboost_d6': 0.167, 'catboost_d4': 0.167}


## 9. Формирование submission

In [16]:
if stack_score > greedy_score:
    final_scores = blending.fit_stacking(oof_ranks, oof_y, test_ranks)
    chosen = f'стекинг (OOF {stack_score:.5f})'
else:
    final_scores = greedy_weights @ test_ranks
    chosen = f'жадный отбор (OOF {greedy_score:.5f})'

submission = pd.DataFrame({
    'lead_id': test['lead_id'],
    'score': blending.normalize_scores(final_scores),
})

assert list(submission.columns) == ['lead_id', 'score']
assert set(submission['lead_id']) == set(sample_submission['lead_id'])
assert submission['score'].between(0, 1).all()
assert not submission['score'].isna().any()

submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {submission.shape}, выбран {chosen}')
submission.head()

submission.csv: (4306, 2), выбран жадный отбор (OOF 0.68777)


,lead_id,score
0,lead_97e409eb8f8c8246,0.117002
1,lead_55310edb4489f9e9,0.668101
2,lead_e7f653a2c6a7eee8,0.942255
3,lead_22f8e1cfc487ac20,0.126032
4,lead_48b638b839abfac3,0.324885


## 10. Значимость признаков

Контрольная проверка: в топе должны быть признаки поведения из `events.csv`,
на которых и держится основной прирост качества.

In [17]:
booster = lgb.train(
    {**primary_params, 'objective': 'binary', 'metric': 'None', 'verbose': -1,
     'seed': config.SEED, 'feature_pre_filter': False},
    lgb.Dataset(train[FEATURES], y, categorical_feature=CATEGORICAL,
                params={'feature_pre_filter': False}),
    300,
)
importance = (pd.DataFrame({'feature': FEATURES, 'gain': booster.feature_importance('gain')})
              .sort_values('gain', ascending=False)
              .head(15)
              .reset_index(drop=True))
importance

,feature,gain
0,ev_nctx,10336.135052
1,ev_ctx_c03,6631.189763
2,ev_hi_3d,6022.617428
3,ev_favorite_rec_h,5337.058601
4,seller_page_views_14d,5120.936542
5,search_views_90d,4159.546378
6,seller_page_views_7d,4148.035713
7,seller_page_views_30d,3863.978696
8,lead_source,3309.672040
9,ev_price_range,3173.975538
